In [54]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jayjoshi37/sleep-screen-time-and-stress-analysis/sleep_mobile_stress_dataset_15000.csv


In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

# Load the dataset - use the path from the file listing above
# Based on your earlier screenshot, it's likely this path:
df = pd.read_csv('/kaggle/input/datasets/jayjoshi37/sleep-screen-time-and-stress-analysis/sleep_mobile_stress_dataset_15000.csv')

print("="*80)
print("DATA LOADED SUCCESSFULLY!!")
print("="*80)
print(f"Dataset Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nColumn Names:")
print(df.columns.tolist())

# Create a clean copy 
df_clean = df.copy()
print("Created df_clean as a copy of the original data")
print(f"df_clean shape: {df_clean.shape}")

DATA LOADED SUCCESSFULLY!!
Dataset Shape: (15000, 13)

First 5 rows:
   user_id  age  gender         occupation  daily_screen_time_hours  \
0        1   56  Female           Designer                     3.26   
1        2   46  Female            Teacher                     1.85   
2        3   32  Female           Designer                     3.04   
3        4   25    Male  Software Engineer                     9.00   
4        5   38  Female            Teacher                     3.52   

   phone_usage_before_sleep_minutes  sleep_duration_hours  \
0                                86                  5.31   
1                                32                  7.36   
2                               107                  4.50   
3                                36                  6.68   
4                                56                  7.57   

   sleep_quality_score  stress_level  caffeine_intake_cups  \
0                 7.72          3.49                     0   
1            

In [56]:
print("="*80)
print("DataSet Info")
print("="*80)
print(df.info())

#check on missing values
print(f"\nMissing Values:")
missing_df = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count' : df.isnull().sum().values,
    'Missing_Percntage' : (df.isnull().sum().values/len(df)*100).round(2)
})
missing_df = missing_df[missing_df['Missing_Count']>0].sort_values('Missing_Count',ascending=False)
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("No Missing Values Found.")
print("="*80)
#check duplicates
print(f"\nDuplicates Values: {df.duplicated().sum()}")

print(f"\nStatistical Summary:")
print(df.describe())


before_count = len(df_clean)
duplicates = df_clean.duplicated().sum()

if duplicates > 0:
    print(f"Found {duplicates} duplicate rows ({duplicates/len(df_clean)*100:.1f}%)")
    
    # Show sample of duplicates
    print("\nSample of duplicate rows:")
    duplicate_samples = df_clean[df_clean.duplicated(keep=False)].head(6)
    print(duplicate_samples)
    
    # Remove duplicates
    df_clean = df_clean.drop_duplicates(keep='first')
    print(f"\nRemoved {duplicates} duplicates")
    print(f"Rows before: {before_count}, After: {len(df_clean)}")
else:
    print("No duplicates found!")
    
print(f"\n**Data types:")
print(df.dtypes)
print(f"\n Unique Values in each column:")
for col in df.columns:
    n_unique = df[col].nunique()
    print(f"{col}:{n_unique} unique values")
    if n_unique < 20:
        print(f"Values: {df[col].unique()[:10]}")
print("="*80)

#Faeture Engineering
if 'stress_level' in df_clean.columns or any('stress' in col.lower() for col in df_clean.columns):
    stress_col = [col for col in df_clean.columns if 'stress' in col.lower()][0]
    print(f"\nCreating stress categories from: {stress_col}")
    
    def categorize_stress(level):
        if level < 3:
            return 'Low Stress'
        elif level < 6:
            return 'Medium Stress'
        else:
            return 'High Stress'
    
    df_clean['Stress_Category'] = df_clean[stress_col].apply(categorize_stress)
    print(f"Created Stress_Category")
    print(df_clean['Stress_Category'].value_counts())



DataSet Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   user_id                           15000 non-null  int64  
 1   age                               15000 non-null  int64  
 2   gender                            15000 non-null  object 
 3   occupation                        15000 non-null  object 
 4   daily_screen_time_hours           15000 non-null  float64
 5   phone_usage_before_sleep_minutes  15000 non-null  int64  
 6   sleep_duration_hours              15000 non-null  float64
 7   sleep_quality_score               15000 non-null  float64
 8   stress_level                      15000 non-null  float64
 9   caffeine_intake_cups              15000 non-null  int64  
 10  physical_activity_minutes         15000 non-null  int64  
 11  notifications_received_per_day    15000 non-null  int6

In [57]:
print("="*80)
print(f"Final shape: {df_clean.shape}")
print(f"\nFinal columns: {df_clean.columns.tolist()}")
print(f"\nMissing values after cleaning:")
print(df_clean.isnull().sum())

# Save to CSV
df_clean.to_csv('/kaggle/working/sleep_stress_cleaned.csv', index=False)
print("\nSaved: sleep_stress_cleaned.csv")

# Create Excel file
with pd.ExcelWriter('/kaggle/working/sleep_stress_dashboard.xlsx', engine='openpyxl') as writer:
    df_clean.to_excel(writer, sheet_name='Master_Data', index=False)

print("Saved: sleep_stress_dashboard.xlsx")

print("\nFiles saved in /kaggle/working/:")
!ls -la /kaggle/working/

print("\n" + "="*80)
print("PREPROCESSING COMPLETE!")
print("="*80)

Final shape: (15000, 14)

Final columns: ['user_id', 'age', 'gender', 'occupation', 'daily_screen_time_hours', 'phone_usage_before_sleep_minutes', 'sleep_duration_hours', 'sleep_quality_score', 'stress_level', 'caffeine_intake_cups', 'physical_activity_minutes', 'notifications_received_per_day', 'mental_fatigue_score', 'Stress_Category']

Missing values after cleaning:
user_id                             0
age                                 0
gender                              0
occupation                          0
daily_screen_time_hours             0
phone_usage_before_sleep_minutes    0
sleep_duration_hours                0
sleep_quality_score                 0
stress_level                        0
caffeine_intake_cups                0
physical_activity_minutes           0
notifications_received_per_day      0
mental_fatigue_score                0
Stress_Category                     0
dtype: int64

Saved: sleep_stress_cleaned.csv
Saved: sleep_stress_dashboard.xlsx

Files saved in